In [1]:
import logging

import numpy as np
import optimistix as optx
from atmodeller import (
    EquilibriumModel,
    Planet,
    ChemicalSpecies,
    SpeciesNetwork,
    debug_logger,
    earth_oceans_to_hydrogen_mass,
    SolverParameters,
)
from atmodeller.eos import get_eos_models
from atmodeller.solubility import get_solubility_models
from atmodeller.thermodata import IronWustiteBuffer
from atmodeller.interfaces import FugacityConstraintProtocol, ThermodynamicStateProtocol
from atmodeller import debug_logger
from atmodeller.classes import EquilibriumModel
from atmodeller.containers import (
    ChemicalSpecies,
    FixedFugacityConstraint,
    SpeciesNetwork,
    ThermodynamicState,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
from matplotlib.patches import Patch

from scipy.interpolate import interp1d
from scipy.optimize import fsolve, root_scalar
import os

from matplotlib.ticker import ScalarFormatter

formatter = ScalarFormatter(useMathText=False)
formatter.set_scientific(False)
formatter.set_useOffset(False)
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

logger = debug_logger() #you can dump it in file 
#logger.setLevel(logging.INFO) #you could set it to error

logger.setLevel(logging.DEBUG)  

#gamma 0.05 0.01 guillot 2010 f 1/4 tau compute 

Atmodeller initialized with double precision (float64)


# Definitions (run these first)

In [2]:
color_tot = "black"
color_H2 = "orange"
color_H2O = "blue"
color_SiH4 = "red"
color_SiO2 = "yellow"
color_SiO = "brown"
color_O2 = "limegreen"
color_CO2 = "cyan"
color_CO = "magenta"
color_CH4 = "purple"
color_N2 = "green"
color_NH3 = "pink"
color_MgO = "darkred"
color_He = "gray"
color_H = "orange"
color_C = "cyan"
color_N = "green"
color_O = "blue"
color_Si = "red"
color_Mg = "lime"
color_MgH2 = "gold"
color_MgH = "teal"
color_MgOH = "indigo"
color_Ccr = "dimgray"

In [3]:
SiO2_mantlemasspercent_palme14: float = 45.4
total_mantlemasspercent_palme14: float = 98.41
core_mass_fraction: float = 0.327  # typical values used are between 0.325 - 0.33
Si_massfraction_palme14: float = round(
    SiO2_mantlemasspercent_palme14
    / total_mantlemasspercent_palme14
    * (1 - core_mass_fraction)
    / (28.0855 + 2 * 15.999)
    * 28.0855,
    3,
)

O_massfraction_palme14: float = Si_massfraction_palme14 / 28.0855 * 2 * 15.999
# Lodders et al. (2009) Springer Book Chapter - Table 8 (in wt%)

H_masspercent_lodders09: float = 73.9
He_masspercent_lodders09: float = 24.69
C_masspercent_lodders09: float = 0.22
N_masspercent_lodders09: float = 0.07
O_masspercent_lodders09: float = 0.63
Si_masspercent_lodders09: float = 0.07
Mg_masspercent_lodders09: float = 0.06

# Lodders et al. (2009) Springer Book Chapter - Table 6 (log-normalalized abundances relative to H)

H_logN: float = 12
He_logN: float = 10.93
C_logN: float = 8.39
N_logN: float = 7.86
O_logN: float = 8.73
Si_logN: float = 7.53
Mg_logN: float = 7.54 

In [4]:
eos_models = get_eos_models()
sol_models = get_solubility_models()

H2O_g = ChemicalSpecies.create_gas("H2O")
H2O_gs = ChemicalSpecies.create_gas("H2O", solubility=sol_models["H2O_peridotite_sossi23"])
H2O_rgs = ChemicalSpecies.create_gas(
    "H2O",
    activity=eos_models["H2O_cork_holland98"],
    solubility=sol_models["H2O_peridotite_sossi23"],
)
H2O_rg = ChemicalSpecies.create_gas(
    "H2O",
    activity=eos_models["H2O_cork_holland98"],
)

H2_g = ChemicalSpecies.create_gas("H2")
H2_gs = ChemicalSpecies.create_gas("H2", solubility=sol_models["H2_basalt_hirschmann12"])
H2_rgs = ChemicalSpecies.create_gas(
    "H2", activity=eos_models["H2_chabrier21"], solubility=sol_models["H2_basalt_hirschmann12"]
)
H2_rg = ChemicalSpecies.create_gas(
    "H2", activity=eos_models["H2_chabrier21"]
)

O2_g = ChemicalSpecies.create_gas("O2")
O2_rg = ChemicalSpecies.create_gas("O2", activity=eos_models["O2_cs_shi92"])

OSi_g = ChemicalSpecies.create_gas("OSi")
OSi_rg = ChemicalSpecies.create_gas("OSi", activity=eos_models["OSi_rk49_connolly16"])

H4Si_g = ChemicalSpecies.create_gas("H4Si")
H4Si_rg = ChemicalSpecies.create_gas("H4Si", activity=eos_models["H4Si_wang18"])

O2Si_cd = ChemicalSpecies.create_condensed("O2Si", state="cd")

C_cd = ChemicalSpecies.create_condensed("C", state="cd")
CSi_cd = ChemicalSpecies.create_condensed("CSi", state="cd")
Si_cd = ChemicalSpecies.create_condensed("Si", state="cd")
N4Si3_cd = ChemicalSpecies.create_condensed("N4Si3", state="cd")
H2O_cd = ChemicalSpecies.create_condensed("H2O", state="cd")

CO2_g = ChemicalSpecies.create_gas("CO2")
CO2_gs = ChemicalSpecies.create_gas("CO2", solubility=sol_models["CO2_basalt_dixon95"])
CO2_rgs = ChemicalSpecies.create_gas(
    "CO2", solubility=sol_models["CO2_basalt_dixon95"], activity=eos_models["CO2_cs_shi92"]
)
CO2_rg = ChemicalSpecies.create_gas(
    "CO2",activity=eos_models["CO2_cs_shi92"]
)

CO_g = ChemicalSpecies.create_gas("CO")
CO_gs = ChemicalSpecies.create_gas("CO", solubility=sol_models["CO_basalt_yoshioka19"])
CO_rgs = ChemicalSpecies.create_gas(
    "CO", solubility=sol_models["CO_basalt_yoshioka19"], activity=eos_models["CO_cs_shi92"]
)
CO_rg = ChemicalSpecies.create_gas(
    "CO", activity=eos_models["CO_cs_shi92"]
)

CH4_g = ChemicalSpecies.create_gas("CH4")
CH4_gs = ChemicalSpecies.create_gas("CH4", solubility=sol_models["CH4_basalt_ardia13"])
CH4_rgs = ChemicalSpecies.create_gas(
    "CH4", solubility=sol_models["CH4_basalt_ardia13"], activity=eos_models["CH4_cs_shi92"]
)
CH4_rg = ChemicalSpecies.create_gas(
    "CH4", activity=eos_models["CH4_cs_shi92"]
)

N2_g = ChemicalSpecies.create_gas("N2")
N2_gs = ChemicalSpecies.create_gas("N2", solubility=sol_models["N2_basalt_libourel03"])
N2_rgs = ChemicalSpecies.create_gas(
    "N2",solubility=sol_models["N2_basalt_libourel03"], activity=eos_models["N2_cs_saxena87"]
)
N2_rg = ChemicalSpecies.create_gas(
    "N2", activity=eos_models["N2_cs_saxena87"]
)


NH3_g = ChemicalSpecies.create_gas("H3N")
NH3_rg = ChemicalSpecies.create_gas("H3N", activity=eos_models["H3N_rk49_reid87"])

He_g = ChemicalSpecies.create_gas("He")
He_gs = ChemicalSpecies.create_gas("He", solubility=sol_models["He_basalt_jambon86"])
He_rgs = ChemicalSpecies.create_gas(
    "He", solubility=sol_models["He_basalt_jambon86"], activity=eos_models["He_chabrier21"]
)
He_rg = ChemicalSpecies.create_gas(
    "He", activity=eos_models["He_chabrier21"]
)


species_HOSi_magma_nosol_ideal = SpeciesNetwork((H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd))
species_HOSi_magma_sol_ideal = SpeciesNetwork((H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd))
species_HOSi_magma_sol_real = SpeciesNetwork((H2O_rgs, H2_rgs, O2_rg, OSi_rg, H4Si_rg, O2Si_cd))
species_HOSi_atm_sol_real = SpeciesNetwork((H2O_g, H2_g, O2_g, H2O_cd,OSi_rg, H4Si_rg))

species_HHeOSi_magma_nosol_ideal = SpeciesNetwork(
    (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd, He_g)
)
species_HHeOSi_magma_sol_ideal = SpeciesNetwork(
    (H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd,He_gs)
)
species_HHeOSi_magma_sol_real = SpeciesNetwork(
    (
        H2O_rgs,
        H2_rgs,
        O2_rg,
        OSi_rg,
        H4Si_rg,
        O2Si_cd,
        He_rgs,
    )
)

species_HCOSi_magma_nosol_ideal = SpeciesNetwork(
    (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_g, CO_g, CH4_g)
)
species_HCOSi_magma_sol_ideal = SpeciesNetwork(
    (H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_gs, CO_gs, CH4_gs)
)
species_HCOSi_magma_sol_real = SpeciesNetwork(
    (
        H2O_rgs,
        H2_rgs,
        O2_rg,
        OSi_rg,
        H4Si_rg,
        O2Si_cd,
        CO2_rgs,
        CO_rgs,
        CH4_rgs,
    )
)

species_HHeCNOSi_magma_nosol_ideal = SpeciesNetwork(
    (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_g, CO_g, CH4_g, N2_g, NH3_g, He_g)
)
species_HHeCNOSi_magma_sol_ideal = SpeciesNetwork(
    (H2O_gs, H2_gs, O2_g, OSi_g, H4Si_g, O2Si_cd, CO2_gs, CO_gs, CH4_gs, N2_gs, NH3_g, He_gs)
)
species_HHeCNOSi_magma_sol_real = SpeciesNetwork(
    (
        H2O_rgs,
        H2_rgs,
        O2_rg,
        OSi_rg,
        H4Si_rg,
        O2Si_cd,
        CO2_rgs,
        CO_rgs,
        CH4_rgs,
        N2_rgs,
        NH3_rg,
        He_rgs,
    )
)

species_HHeCNOSi_atmosphere = SpeciesNetwork(
    (H2O_rg, H2_rg, O2_rg, OSi_rg, H4Si_rg, H2O_cd, C_cd, CSi_cd, Si_cd, N4Si3_cd,
     CO2_rg, CO_rg, CH4_rg, N2_rg, NH3_rg, He_rgs)
)
#species_HHeCNOSi_atmosphere = SpeciesNetwork(
#    (H2O_g, H2_g, O2_g, OSi_g, H4Si_g, O2Si_cd, C_cd, CSi_cd, Si_cd, N4Si3_cd,
#     CO2_g, CO_g, CH4_g, N2_g, NH3_g, He_g)
#)

In [5]:
number_of_realisations = 100

# Overall structure 
R_earth = 6.371e6
R_uranus = 3.939 * R_earth
M_earth = 5.972e24 #kg
M_uranus = 14.536 * M_earth

#fO2_log10_shifts = [-1]
fO2_log10_shifts = np.linspace(-6, 0, num=30)  # IW-6 to IW
fO2_log10_shifts = [-6,-5,-4,-3,-2,-1,0]
hmps = np.logspace(-1,0, num=50)  # wt% H
h_kgs = hmps / 100 * M_uranus  # kg

si_kgs_solar = h_kgs * Si_masspercent_lodders09 / H_masspercent_lodders09
mg_kgs_solar = h_kgs * Mg_masspercent_lodders09 / H_masspercent_lodders09
o_kgs_solar = h_kgs * O_masspercent_lodders09 / H_masspercent_lodders09
c_kgs_solar = h_kgs * C_masspercent_lodders09 / H_masspercent_lodders09
n_kgs_solar = h_kgs * N_masspercent_lodders09 / H_masspercent_lodders09
he_kgs_solar = h_kgs * He_masspercent_lodders09 / H_masspercent_lodders09


rho_centres = [2500,3000,2000,1500,1000]
rho_centres = [3000]
rho_centre = 3000
print(rho_centres)
R_core = 0.7*R_uranus
Lowerbound_M_core = [
    4*np.pi*(R_core**3)*rho/3
    for rho in rho_centres
]
M_cores = Lowerbound_M_core
print(M_cores)
print(10*M_earth)

[3000]
[6.812158460855618e+25]
5.972000000000001e+25


In [12]:
# Initialize EquilibriumModel objects 
model_magma_nosol_ideal = EquilibriumModel(species_HOSi_magma_nosol_ideal)
model_magma_sol_ideal = EquilibriumModel(species_HOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HOSi_magma_sol_real)

number_of_realisations = 50
mantle_melt_fraction = 1
init_metallicitys = [1]
for i in range(len(rho_centres)):
    rho_centre = rho_centres[i]
    M_core = M_cores[i]
    fugacity_constraints = {O2_rg.name: IronWustiteBuffer(fO2_log10_shifts)}
    for fO2_log10_shift in fO2_log10_shifts:
        shift = f"{fO2_log10_shift:.2g}"
        planet = Planet(
            temperature=2914,
            planet_mass=11.385*M_earth,
            mantle_melt_fraction=1,
            surface_radius=1.96*R_earth,
        )
        si_kg_magma = Si_massfraction_palme14 * 11.385*M_earth
        si_kg_magma = Si_massfraction_palme14 * M_core
        for init_metallicity in init_metallicitys:
            mass_constraints = {
                "H": h_kgs,
                "Si": init_metallicity * si_kgs_solar + mantle_melt_fraction * si_kg_magma,
            }
            # Magma - No solubility - Ideal Gas
            model_magma_nosol_ideal.solve(
                state=planet,
                mass_constraints=mass_constraints,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=20)
            )
            output_magma_nosol_ideal = model_magma_nosol_ideal.output
            # output_magma_nosol_ideal.quick_look()
            output_magma_nosol_ideal.to_excel(
                f"./TEST"
            )
            output_magma_nosol_ideal.to_pickle(
                f"./TEST"
            )
    
            # Magma - Solubility - Ideal Gas
            initial_log_number_moles = output_magma_nosol_ideal.log_number_moles
            model_magma_sol_ideal.solve(
                state=planet,
                mass_constraints=mass_constraints,
                initial_log_number_moles=initial_log_number_moles,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=20)
            )
            output_magma_sol_ideal = model_magma_sol_ideal.output
            # output_magma_sol_ideal.quick_look()
            output_magma_sol_ideal.to_excel(
                f"./TEST"
            )
            output_magma_sol_ideal.to_pickle(
                f"./TEST"
            )
            # Magma - Solubility - Real Gas
            initial_log_number_moles = output_magma_sol_ideal.log_number_moles
            model_magma_sol_real.solve(
                state=planet,
                mass_constraints=mass_constraints,
                initial_log_number_moles=initial_log_number_moles,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=50)
            )
            output_magma_sol_real = model_magma_sol_real.output
            output_magma_sol_real.quick_look()
            output_magma_sol_real.to_excel(
                f"./TEST"
            )
            output_magma_sol_real.to_pickle(
                f"./TEST"
            )

[12:12:04 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[12:12:04 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[12:12:04 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2O_g = 2.0 H2_g + 1.0 O2_g',
 1: '3.0 H2_g + 1.0 OSi_g = 1.0 H2O_g + 1.0 H4Si_g',
 2: '1.0 H2O_g + 1.0 OSi_g = 1.0 H2_g + 1.0 O2Si_cd'}
[12:12:04 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, SolubilityPowerLaw', 'H2_g: IdealGas, SolubilityPowerLawLog10', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility')
[12:12:04 - atmodeller.classes             - INFO     ] - Thermodynamic

IndexError: list index out of range

In [11]:
# Initialize EquilibriumModel objects 
model_magma_nosol_ideal = EquilibriumModel(species_HCOSi_magma_nosol_ideal)
model_magma_sol_ideal = EquilibriumModel(species_HCOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HCOSi_magma_sol_real)

number_of_realisations = 50
fO2_log10_shifts = [-4,-3,-2,-1,0]
mantle_melt_fraction = 1
init_metallicitys = [1]
rho_centres = [3000,3000,3000]
surf_fraction = [0.6]
for i in range(len(rho_centres)):
    rho_centre = rho_centres[i]
    M_core = M_cores[0]
    fugacity_constraints = {O2_rg.name: IronWustiteBuffer(fO2_log10_shifts)}
    for fO2_log10_shift in fO2_log10_shifts:
        shift = f"{fO2_log10_shift:.2g}"
        planet = Planet(
            temperature=2914,
            planet_mass=11.385*M_earth,
            mantle_melt_fraction=1,
            surface_radius=1.96*R_earth,
        )
        si_kg_magma = Si_massfraction_palme14 * 11.385*M_earth

        for init_metallicity in init_metallicitys:
            mass_constraints = {
                "H": h_kgs,
#                "He": he_kgs_solar,
                "C": init_metallicity * c_kgs_solar,
#                "N": init_metallicity * n_kgs_solar,
                "Si": init_metallicity * si_kgs_solar + mantle_melt_fraction * si_kg_magma,
                }
            # Magma - No solubility - Ideal Gas
            model_magma_nosol_ideal.solve(
                state=planet,
                mass_constraints=mass_constraints,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=50)
            )
            output_magma_nosol_ideal = model_magma_nosol_ideal.output
            # output_magma_nosol_ideal.quick_look()
            output_magma_nosol_ideal.to_excel(
                f"./TEST"
            )
            output_magma_nosol_ideal.to_pickle(
                f"./TEST"
            )
            # Magma - Solubility - Ideal Gas
            initial_log_number_moles = output_magma_nosol_ideal.log_number_moles
            model_magma_sol_ideal.solve(
                state=planet,
                mass_constraints=mass_constraints,
                initial_log_number_moles=initial_log_number_moles,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=20)
            )
            output_magma_sol_ideal = model_magma_sol_ideal.output
            # output_magma_sol_ideal.quick_look()
            output_magma_sol_ideal.to_excel(
                f"./TEST"
            )
            output_magma_sol_ideal.to_pickle(
                f"./TEST"
            )
            # Magma - Solubility - Real Gas
            initial_log_number_moles = output_magma_sol_ideal.log_number_moles
            model_magma_sol_real.solve(
                state=planet,
                mass_constraints=mass_constraints,
                initial_log_number_moles=initial_log_number_moles,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=20)
            )
            output_magma_sol_real = model_magma_sol_real.output
            output_magma_sol_real.quick_look()
            output_magma_sol_real.to_excel(
                f"./TEST"
            )
            output_magma_sol_real.to_pickle(
                f"./TEST"
            )

[13:14:42 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility', 'CO2_g: IdealGas, NoSolubility', 'CO_g: IdealGas, NoSolubility', 'CH4_g: IdealGas, NoSolubility')
[13:14:42 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[13:14:42 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2_g + 1.0 OSi_g = 0.5 O2_g + 1.0 H4Si_g',
 1: '0.5 O2_g + 1.0 OSi_g = 1.0 O2Si_cd',
 2: '1.0 H2_g + 0.5 O2_g = 1.0 H2O_g',
 3: '1.0 CO2_g = 0.5 O2_g + 1.0 CO_g',
 4: '2.0 H2_g + 1.0 CO2_g = 1.0 O2_g + 1.0 CH4_g'}
[13:14:42 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, SolubilityPowerLaw', 'H2_g: IdealGas, SolubilityPowerLawLog10', 'O2_g: IdealGas, NoSolubility', 'OSi_g: 

In [11]:
# Initialize EquilibriumModel objects 
model_magma_nosol_ideal = EquilibriumModel(species_HHeCNOSi_magma_nosol_ideal)
model_magma_sol_ideal = EquilibriumModel(species_HHeCNOSi_magma_sol_ideal)
model_magma_sol_real = EquilibriumModel(species_HHeCNOSi_magma_sol_real)

number_of_realisations = 50
mantle_melt_fraction = 1
init_metallicitys = [1]
fO2_log10_shifts = [-6,-5,-4,-3,-2,-1,0]
rho_centres = [3000]
surf_fraction = [0.6]
for i in range(len(rho_centres)):
    rho_centre = rho_centres[i]
    M_core = M_cores[0]
    fugacity_constraints = {O2_rg.name: IronWustiteBuffer(fO2_log10_shifts)}
    for fO2_log10_shift in fO2_log10_shifts:
        shift = f"{fO2_log10_shift:.2g}"
        print(shift)
        planet = Planet(
            temperature=1080,
            planet_mass=10.641091*M_earth,
            mantle_melt_fraction=0,
            surface_radius=1.91692*R_earth,
        )
        si_kg_magma = Si_massfraction_palme14 * 10.641091*M_earth

        for init_metallicity in init_metallicitys:
            mass_constraints = {
                "H": h_kgs,
                "He": he_kgs_solar,
                "C": init_metallicity * c_kgs_solar,
                "N": init_metallicity * n_kgs_solar,
                "Si": init_metallicity * si_kgs_solar + mantle_melt_fraction * si_kg_magma,
                }
            # Magma - No solubility - Ideal Gas
            model_magma_nosol_ideal.solve(
                state=planet,
                mass_constraints=mass_constraints,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=50)
            )
            output_magma_nosol_ideal = model_magma_nosol_ideal.output
            # output_magma_nosol_ideal.quick_look()
            output_magma_nosol_ideal.to_excel(
                f"./TEST"
            )
            output_magma_nosol_ideal.to_pickle(
                f"./TEST"
            )
            # Magma - Solubility - Ideal Gas
            initial_log_number_moles = output_magma_nosol_ideal.log_number_moles
            model_magma_sol_ideal.solve(
                state=planet,
                mass_constraints=mass_constraints,
                initial_log_number_moles=initial_log_number_moles,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=20)
            )
            output_magma_sol_ideal = model_magma_sol_ideal.output
            # output_magma_sol_ideal.quick_look()
            output_magma_sol_ideal.to_excel(
                f"./TEST"
            )
            output_magma_sol_ideal.to_pickle(
                f"./TEST"
            )
            # Magma - Solubility - Real Gas
            initial_log_number_moles = output_magma_sol_ideal.log_number_moles
            model_magma_sol_real.solve(
                state=planet,
                mass_constraints=mass_constraints,
                initial_log_number_moles=initial_log_number_moles,
                fugacity_constraints={O2_rg.name: IronWustiteBuffer(np.full(number_of_realisations, fO2_log10_shift))},
                solver_parameters=SolverParameters(multistart=20)
            )
            output_magma_sol_real = model_magma_sol_real.output
            output_magma_sol_real.quick_look()
            output_magma_sol_real.to_excel(
                f"./TEST"
            )
            output_magma_sol_real.to_pickle(
                f"./TEST"
            )

[16:26:51 - atmodeller.classes             - INFO     ] - species_network = ('H2O_g: IdealGas, NoSolubility', 'H2_g: IdealGas, NoSolubility', 'O2_g: IdealGas, NoSolubility', 'OSi_g: IdealGas, NoSolubility', 'H4Si_g: IdealGas, NoSolubility', 'O2Si_cd: CondensateActivity, NoSolubility', 'CO2_g: IdealGas, NoSolubility', 'CO_g: IdealGas, NoSolubility', 'CH4_g: IdealGas, NoSolubility', 'N2_g: IdealGas, NoSolubility', 'H3N_g: IdealGas, NoSolubility', 'He_g: IdealGas, NoSolubility')
[16:26:51 - atmodeller.classes             - INFO     ] - Thermodynamic data requires temperatures between 200 K and 6000 K
[16:26:52 - atmodeller.classes             - INFO     ] - reactions = {0: '2.0 H2_g + 0.5 O2Si_cd = 1.0 H2O_g + 0.5 H4Si_g',
 1: '0.5 H4Si_g + 1.0 CO2_g = 1.0 H2_g + 0.5 O2Si_cd + 1.0 CO_g',
 2: '1.0 H4Si_g + 1.0 CO2_g = 1.0 O2Si_cd + 1.0 CH4_g',
 3: '0.5 H4Si_g + 0.5 O2Si_cd = 1.0 H2_g + 1.0 OSi_g',
 4: '1.5 H2_g + 0.5 N2_g = 1.0 H3N_g',
 5: '2.0 H2_g + 1.0 O2Si_cd = 1.0 O2_g + 1.0 H4Si_g'}


-6


[16:27:39 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 50 (100.00%) successful model(s)
[16:27:39 - atmodeller.classes             - INFO     ] - Multistart summary: 50 (100.00%) models(s) required 1 attempt(s)
[16:27:39 - atmodeller.classes             - INFO     ] - Solver steps (max) = 36
[16:27:39 - atmodeller.output_core         - DEBUG    ] - Creating Output
[16:27:39 - atmodeller.output              - INFO     ] - Writing output to excel
[16:27:39 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[16:27:39 - atmodeller.output_core         - INFO     ] - Computing asdict output
[16:27:40 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[16:27:41 - atmodeller.output              - INFO     ] - Output written to TEST.xlsx
[16:27:41 - atmodeller.output              - INFO     ] - Writing output to pickle
[16:27:41 - atmodeller.output_core         - DEBUG    ] - Returning cached t

-5


[16:36:49 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 50 (100.00%) successful model(s)
[16:36:49 - atmodeller.classes             - INFO     ] - Multistart summary: 50 (100.00%) models(s) required 1 attempt(s)
[16:36:49 - atmodeller.classes             - INFO     ] - Solver steps (max) = 36
[16:36:49 - atmodeller.output_core         - DEBUG    ] - Creating Output
[16:36:51 - atmodeller.output              - INFO     ] - Writing output to excel
[16:36:51 - atmodeller.output_core         - INFO     ] - Computing to_dataframes output
[16:36:51 - atmodeller.output_core         - INFO     ] - Computing asdict output
[16:36:52 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[16:36:55 - atmodeller.output              - INFO     ] - Output written to TEST.xlsx
[16:36:55 - atmodeller.output              - INFO     ] - Writing output to pickle
[16:36:55 - atmodeller.output_core         - DEBUG    ] - Returning cached t

-4


... (4 outputs left out)

[16:38:44 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[16:38:46 - atmodeller.output              - INFO     ] - Output written to TEST.xlsx
[16:38:46 - atmodeller.output              - INFO     ] - Writing output to pickle
[16:38:46 - atmodeller.output_core         - DEBUG    ] - Returning cached to_dataframes output
[16:38:46 - atmodeller.output              - INFO     ] - Output written to TEST.pkl
[16:38:46 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 50 (100.00%) successful model(s)
[16:38:46 - atmodeller.classes             - INFO     ] - Multistart summary: 50 (100.00%) models(s) required 1 attempt(s)
[16:38:46 - atmodeller.classes             - INFO     ] - Solver steps (max) = 3
[16:38:46 - atmodeller.output_core         - DEBUG    ] - Creating Output
[16:38:46 - atmodeller.output              - INFO     ] - Writing output to excel
[16:38:46 - atmodeller.output_core         - INFO     ] - Computing

-1


[16:39:02 - atmodeller.output_core         - DEBUG    ] - Found O2_g so back-computing log10 shift for fO2
[16:39:04 - atmodeller.output              - INFO     ] - Output written to TEST.xlsx
[16:39:04 - atmodeller.output              - INFO     ] - Writing output to pickle
[16:39:04 - atmodeller.output_core         - DEBUG    ] - Returning cached to_dataframes output
[16:39:04 - atmodeller.output              - INFO     ] - Output written to TEST.pkl
[16:39:04 - atmodeller.classes             - INFO     ] - Solve (robust) complete: 50 (100.00%) successful model(s)
[16:39:04 - atmodeller.classes             - INFO     ] - Multistart summary: 50 (100.00%) models(s) required 1 attempt(s)
[16:39:04 - atmodeller.classes             - INFO     ] - Solver steps (max) = 3
[16:39:04 - atmodeller.output_core         - DEBUG    ] - Creating Output
[16:39:04 - atmodeller.output              - INFO     ] - Writing output to excel
[16:39:04 - atmodeller.output_core         - INFO     ] - Computing

ValueError: INTERNAL: CpuCallback error calling callback: Traceback (most recent call last):
  File "/home/mathias/atmodeller/.venv/lib/python3.12/site-packages/jax/_src/callback.py", line 784, in _wrapped_callback
  File "/home/mathias/atmodeller/.venv/lib/python3.12/site-packages/jax/_src/callback.py", line 223, in _callback
  File "/home/mathias/atmodeller/.venv/lib/python3.12/site-packages/jax/_src/callback.py", line 95, in pure_callback_impl
  File "/home/mathias/atmodeller/.venv/lib/python3.12/site-packages/jax/_src/callback.py", line 70, in __call__
  File "/home/mathias/atmodeller/.venv/lib/python3.12/site-packages/equinox/_errors.py", line 88, in raises
_EquinoxRuntimeError: A linear solver received non-finite (NaN or inf) input and cannot determine a 
solution. 

This means that you have a bug upstream of Lineax and should check the inputs to 
`lineax.linear_solve` for non-finite values.


--------------------
An error occurred during the runtime of your JAX program! Unfortun